# CNN Lab — Image Classification (Frozen Backbone Workflow) — AI Lab Instructor Notebook

*Deep Learning · Hard*



6 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

# default filter count used across sections
n_filters = 16


## Task 1: Dataset & Setup

# CNN Lab — Image Classification (Frozen Backbone Workflow) — Student Lab

We simulate a transfer-learning workflow **without deep learning frameworks**:
- build a frozen conv feature extractor (backbone)
- train a lightweight classifier head (LogisticRegression)

We compare three feature sets:
- raw pixels
- random conv features
- unsupervised "pretrained" conv features learned from patch k-means

Dataset: `sklearn.datasets.load_digits` (8×8 grayscale, offline).

## Section 0 — Data + low-data labeled split

We create a small labeled training set to simulate the transfer-learning regime.

### Baseline: raw pixels + LogisticRegression

In [ ]:
# No source solution — left as an exercise.

## Task 2: Implement `conv2d_forward` and `maxpool2d_forward`.
*Section: CNN forward primitives (NCHW)*

## Section 1 — CNN forward primitives (NCHW)

We implement forward-only conv + pool to build a frozen feature extractor.

### Task 1.1
Implement `conv2d_forward` and `maxpool2d_forward`.

In [ ]:
$$11

In [ ]:
# Checks
x = rng.normal(size=(2, 1, 8, 8))
w = rng.normal(size=(4, 1, 3, 3))
y0 = conv2d_forward(x, w, b=None, stride=1, padding=0)
check('conv_shape', y0.shape == (2, 4, 6, 6))
y1 = conv2d_forward(x, w, b=np.zeros(4), stride=1, padding=1)
check('conv_shape_pad', y1.shape == (2, 4, 8, 8))
p = maxpool2d_forward(y1, pool=2, stride=2)
check('pool_shape', p.shape == (2, 4, 4, 4))

## Task 3: Implement `extract_conv_features(X_img, filters)` where `filters` has shape `(C_out, 1, 3, 3)`.
*Section: Feature extractor*

## Section 2 — Feature extractor

Pipeline: `conv (3x3, pad=1) → ReLU → maxpool(2)` → flatten.

### Task 2.1
Implement `extract_conv_features(X_img, filters)` where `filters` has shape `(C_out, 1, 3, 3)`.

In [ ]:
def extract_conv_features(X_img, filters):
    x = X_img[:, None, :, :]
    y = conv2d_forward(x, filters, b=None, stride=1, padding=1)
    y = relu(y)
    y = maxpool2d_forward(y, pool=2, stride=2)
    return y.reshape(len(X_img), -1)

n_filters = 16
filters = rng.normal(0.0, 1.0, size=(n_filters, 1, 3, 3))
F_lab_feat = extract_conv_features(X_lab_n, filters)
F_val_feat = extract_conv_features(X_val_n, filters)
print('feature shapes', F_lab_feat.shape, F_val_feat.shape)
check('feat_rows', F_lab_feat.shape[0] == len(X_lab_n) and F_val_feat.shape[0] == len(X_val_n))
check('feat_dim', F_lab_feat.shape[1] == 16 * 4 * 4)

In [ ]:
# Checks
check('feat_rows', F_lab_feat.shape[0] == len(X_lab_n) and F_val_feat.shape[0] == len(X_val_n))
check('feat_dim', F_lab_feat.shape[1] == 16 * 4 * 4)

## Task 4: Sample random filters and evaluate a classifier on top.
*Section: Random conv features*

## Section 3 — Random conv features

### Task 3.1
Sample random filters and evaluate a classifier on top.

In [ ]:
n_filters = 16
rand_filters = rng.normal(0.0, 1.0, size=(n_filters, 1, 3, 3))

F_lab_rand = extract_conv_features(X_lab_n, rand_filters)
F_val_rand = extract_conv_features(X_val_n, rand_filters)

rand_clf = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
rand_clf.fit(F_lab_rand, y_lab)
rand_val = float(rand_clf.score(F_val_rand, y_val))
print('random conv val acc', rand_val)
check('rand_sane', 0.6 <= rand_val <= 1.0)

In [ ]:
# Checks
check('rand_sane', 0.6 <= rand_val <= 1.0)

**Why this works.** ---
## Summary

## Task 5: Implement `extract_patches(X_img, k=3, n_patches)` returning (n_patches, k*k).
*Section: Unsupervised "pretraining": patch k-means*

## Section 4 — Unsupervised "pretraining": patch k-means

We learn filters by clustering many 3×3 patches from unlabeled training images.

```text
unlabeled images
   ↓
extract random 3×3 patches
   ↓
run k-means on patches
   ↓
centroids become conv filters
```

<div style="max-width: 72%; margin: 1.25rem auto;">
  <img src="/images/w8-d3-kmeans-clustering-overview.png" alt="K-means clustering overview" width="1200" />
</div>

### Task 4.1
Implement `extract_patches(X_img, k=3, n_patches)` returning (n_patches, k*k).

### Task 4.2
Implement `kmeans(X, k, n_iter)` returning centroids.

HINT: keep n_patches and n_iter small (digits is tiny).

**Hints**
- keep n_patches and n_iter small (digits is tiny).

In [ ]:
$$12

In [ ]:
# Checks
check('cent_shape', centroids.shape == (n_filters, 9))

**Why this works.** ---
## Summary

## Task 6: Use k-means centroids as filters and evaluate classifier performance.
*Section: Unsupervised "pretraining": patch k-means*

### Task 4.3
Use k-means centroids as filters and evaluate classifier performance.

In [ ]:
n_filters = 16
km_filters = centroids.reshape(n_filters, 1, 3, 3)

F_lab_km = extract_conv_features(X_lab_n, km_filters)
F_val_km = extract_conv_features(X_val_n, km_filters)

km_clf = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
km_clf.fit(F_lab_km, y_lab)
km_val = float(km_clf.score(F_val_km, y_val))
print('kmeans-pretrained conv val acc', km_val)
check('km_sane', 0.6 <= km_val <= 1.0)

In [ ]:
# Checks
check('km_sane', 0.6 <= km_val <= 1.0)

**Why this works.** ---
## Summary